<a href="https://colab.research.google.com/github/tougheye/Data_processing/blob/main/TCS_Differential_Process.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
tcs_backend_folder = "/content/drive/MyDrive/Data/UCOP_Job_Codes_Summary"
shift_on_call_full_df = pd.read_excel(f"{tcs_backend_folder}/UPTE-26-015_Job Summary Report_3.20.26.xlsx", sheet_name="Shift and On Call Rates")
shift_on_call_full_df = shift_on_call_full_df.iloc[:-2]
# shift_on_call_full_df.shape

In [ ]:
job_code_review_full_df = pd.read_excel(f"{tcs_backend_folder}/UPTE-26-015_Job Summary Report_3.20.26.xlsx", sheet_name="Job Code Review")
# job_code_review_full_df.shape

In [ ]:
upte_units = ['HX', 'RX', 'TX']
job_code_review_upte_df = job_code_review_full_df[job_code_review_full_df['Union Code'].isin(upte_units)]
# job_code_review_upte_df.shape

In [ ]:
upte_shift_on_call_df = shift_on_call_full_df.merge(job_code_review_upte_df[['Job Code', 'Union Code']], on= 'Job Code', how='inner').drop_duplicates()
# upte_shift_on_call_df.shape

In [40]:
output_path = "/content/drive/MyDrive/UPTE_Differential_lists"
upte_shift_on_call_df.to_excel(f"{output_path}/upte_systemwide_diff_master_list_0320206.xlsx", index=False)

In [ ]:
business_units = upte_shift_on_call_df['Salary Plan SETID'].unique()
# business_units

In [ ]:
diff_types_list = upte_shift_on_call_df['Earn Code Description'].unique()
# diff_types_list

Process all differential details at the bargaining unit and job title level into dictionaries

In [58]:

diff_details_dict = {}

for bus_unit in business_units:

  curr_bus_unit_df = upte_shift_on_call_df[upte_shift_on_call_df['Salary Plan SETID'] == bus_unit]

  diff_details_dict[bus_unit] = {}

  for barg_unit in curr_bus_unit_df['Union Code'].unique():
    diff_details_dict[bus_unit][barg_unit] = {}

    curr_barg_unit_df = curr_bus_unit_df[curr_bus_unit_df['Union Code'] == barg_unit]
    titles_list = sorted(curr_barg_unit_df['Job Description'].unique())

    for title in titles_list:

      diff_details_dict[bus_unit][barg_unit][title] = {}
      diff_details_dict[bus_unit][barg_unit][title]["Job Title"] = title
      diff_details_dict[bus_unit][barg_unit][title]["Job Code"] = \
      curr_barg_unit_df[curr_barg_unit_df['Job Description'] == title]['Job Code'].values[0]

    # process the differential types
      for diff_type in diff_types_list:
        filter = (curr_barg_unit_df['Earn Code Description'] == diff_type) & \
                (curr_barg_unit_df['Job Description'] == title)
        if not curr_barg_unit_df[filter].empty:
          r = curr_barg_unit_df[filter]['Hourly Rate'].values[0]
          diff_details_dict[bus_unit][barg_unit][title][diff_type] = f'${r:,.2f}'
        else:
          diff_details_dict[bus_unit][barg_unit][title][diff_type] = 0



Create the output excel files

In [57]:
# create the folders (only once)
# DONE!!! NEVER USE IT AGAIN
import os
output_path = "/content/drive/MyDrive/UPTE_Differential_lists"
for bus_unit in business_units:
  os.makedirs(f"{output_path}/{bus_unit}", exist_ok=True)

In [ ]:
for bus_unit in business_units:

  barg_units = ['HX', 'RX', 'TX']

  with pd.ExcelWriter(f"{output_path}/{bus_unit}/{bus_unit}_differential_master_list_0320206.xlsx") as writer:


  # loop throug hteh bargainign units
    for barg_unit in diff_details_dict[bus_unit]:
      print(f"Processing {bus_unit}'s {barg_unit}")

      campus_bu = diff_details_dict[bus_unit][barg_unit]

      curr_df_list = []
      for title in diff_details_dict[bus_unit][barg_unit]:
        df = pd.DataFrame(campus_bu[title], index=[0])
        curr_df_list.append(df)

      curr_df = pd.concat(curr_df_list)
      curr_df.to_excel(writer, sheet_name=barg_unit, index=False)